

## **Crop Prediction using GEE**



> Initialize GEE and Autheticate



In [4]:
import ee
import geemap

In [5]:
ee.Authenticate()
ee.Initialize(project = "ee-leviekytz")

In [6]:
m = geemap.Map(center = [22.62523109,88.49792521], zoom = 15)
m

Map(center=[22.62523109, 88.49792521], controls=(WidgetControl(options=['position', 'transparent_bg'], positio…

###Loading the various fields

In [7]:
fields = ee.FeatureCollection("projects/ee-leviekytz/assets/yield_prediction_dataset")
fields

#Create 500 m buffer in the fields
buffered_fields = fields.map(lambda f: f.buffer(500))
print(buffered_fields.first().getInfo())

#Display the buffered areas
vis_param = {
    "color": "red"
}
m.add_layer(buffered_fields, vis_param, "Farm lands")
m

{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[88.49792362170211, 22.629728468501135], [88.496562044677, 22.62954938532371], [88.49530885913181, 22.62902639214809], [88.49426382547934, 22.628201122821295], [88.49351013000056, 22.627139273505826], [88.49310776271709, 22.62592537179058], [88.49308874347773, 22.62465604692688], [88.49345457585868, 22.623432337305548], [88.49417613071053, 22.622351647760198], [88.49519596761273, 22.62149999690801], [88.49643290858283, 22.62094517140043], [88.49778849952038, 22.620731331589784], [88.49915484515611, 22.620875497537693], [88.50042319446733, 22.62136619471331], [88.50149259411134, 22.622164367029537], [88.50227792204552, 22.62320648464429], [88.50271666256462, 22.624409599477527], [88.50277388367414, 22.62567794647984], [88.5024450202507, 22.626910565627874], [88.50175624064731, 22.62800933821772], [88.50076236655617, 22.62888679780616], [88.49954251081445, 22.62947309383309], [88.49819377987947, 22.629721553217045], [88

Map(center=[22.62523109, 88.49792521], controls=(WidgetControl(options=['position', 'transparent_bg'], positio…

### Importing Sentinel 2 imagery

In [8]:
def mask_s2_clouds(image):
  """Masks clouds in a Sentinel-2 image using the QA band.

  Args:
      image (ee.Image): A Sentinel-2 image.

  Returns:
      ee.Image: A cloud-masked Sentinel-2 image.
  """
  qa = image.select('QA60')

  # Bits 10 and 11 are clouds and cirrus, respectively.
  cloud_bit_mask = 1 << 10
  cirrus_bit_mask = 1 << 11

  # Both flags should be set to zero, indicating clear conditions.
  mask = (
      qa.bitwiseAnd(cloud_bit_mask)
      .eq(0)
      .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
  )

  return image.updateMask(mask).divide(10000)
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
.filterDate('2023-01-01','2023-12-31')
.filterBounds(buffered_fields)
.map(mask_s2_clouds)
)

image = s2.median()

vis_params = {
    'min' : 0.0,
    'max' : 0.3,
    'bands' : ["B4","B3","B2"]
}
m.add_layer(image, vis_params, "Sentinel 2 Images")
m

Map(bottom=3653137.0, center=[22.62523109, 88.49792521], controls=(WidgetControl(options=['position', 'transpa…

### Function that computes for:


*   NDVI,GNDVI,NDWI,SAVI
*   Rainfall, Temperature



In [9]:
def ndvi_add(image):
  ndvi = image.normalizedDifference(["B8","B4"]).rename("NDVI")
  return image.addBands(ndvi)

#Apply the ndvi to the entire Image Collection
s2_ndvi = s2.map(ndvi_add)

In [ ]:
def extract_NDVI(feature):
  #Extracting the date
  date = ee.Date.parse('DD-MM-YYYY', feature.get('date_of_image'))
  start_date = date.advance(-7,'day')
  end_date = date.advance(7,'day')

  #Extracting the first image- compute ndvi from sentinel 2
  imagery = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
  .filterBounds(feature.geometry())
  .filterDate(start_date, end_date)
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first())

  #Getting centroid coordinates for the fields
  centroid = feature.geometry().centroid()
  coords = centroid.coordinates()
  longitude = ee.Number(coords.get(0))
  latitude = ee.Number(coords.get(1))

  #Extracting ndvi
  ndvi = imagery.normalizedDifference(["B8","B4"]).rename("NDVI")

  #Finding the mean ndvi
  ndvi_mean = ndvi.reduceRegion(
      reducer = ee.Reducer.mean(),
      geometry = feature.geometry(),
      scale = 10,
      maxPixels = 1e9
  )

  #Extracting gndvi
  red = imagery.select('B4').divide(10000.0)  #Red band
  green = imagery.select('B3').divide(10000.0)  #Green band
  nir = imagery.select('B8').divide(10000.0)  #NIR band

  gndvi = imagery.expression(
      '(NIR - GREEN) / (NIR + GREEN)',
      {'NIR' : nir,
       'GREEN' : green}
  ).rename('GNDVI')

  gndvi_mean = gndvi.reduceRegion(
      reducer = ee.Reducer.mean(),
      geometry = feature.geometry(),
      scale = 10,
      maxPixels = 1e9
  )

  #Extracting savi
  savi = imagery.expression(
      '((NIR - Red)/(NIR + Red + L)) * (1 + L)',
      {'NIR': nir,
       'Red' : red,
       'L' : 0.5}  #Soil adjustment factor
  ).rename('SAVI')

  savi_mean = savi.reduceRegion(
      reducer = ee.Reducer.mean(),
      geometry = feature.geometry(),
      maxPixels = 1e9,
      scale = 10
  )

  #Extracting NDWI
  ndwi = imagery.expression(
      '(GREEN - NIR)/(GREEN + NIR)',
      {'GREEN': green,
       'NIR': nir}
  ).rename('NDWI')

  ndwi_mean = ndwi.reduceRegion(
      reducer = ee.Reducer.mean(),
      geometry = feature.geometry(),
      scale = 10,
      maxPixels = 1e9
  )

  #Extracting rainfall data
  rainfall = (ee.ImageCollection("UCSB-CHC/CHIRPS/V3/DAILY_SAT")
  .filterDate(start_date, end_date)
  .filterBounds(feature.geometry())
  .sum()
  )

  rainfall_mean = rainfall.reduceRegion(
      reducer = ee.Reducer.mean(),
      geometry = feature.geometry(),
      scale = 5000,
      maxPixels = 1e9
  )



  #Extracting temperature data
  temp = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
  .filterDate(start_date, end_date)
  .filterBounds(feature.geometry())
  .select("temperature_2m")
  .mean())


  temp_mean = temp.reduceRegion(
      reducer = ee.Reducer.mean(),
      geometry = feature.geometry(),
      scale = 5000, # Changed scale from 10000 to 5000
      maxPixels = 1e9
  )

  temp_k = temp_mean.get('temperature_2m')
  # Conditionally convert temperature if not null
  # Use None directly for falseCase to avoid JSON serialization issues with ee.Number(float('nan'))
  temp_celcius = ee.Algorithms.If(temp_k, ee.Number(temp_k).subtract(273.15), None)

  return feature.set({
      'gee_ndvi':ndvi_mean.get('NDVI'),
      'gee_gndvi': gndvi_mean.get('GNDVI'),
      'gee_savi': savi_mean.get('SAVI'),
      'gee_ndwi': ndwi_mean.get('NDWI'),
      'gee_rainfall':rainfall_mean.get('precipitation'),
      'gee_temp': temp_celcius,
      'latitude': latitude,
      'longitude': longitude})

In [19]:
ndvi_fields = buffered_fields.map(extract_NDVI)

ndvi_fields.first().getInfo()

AttributeError: 'Feature' object has no attribute 'coordinates'

### Export to Pandas

In [12]:
features = ndvi_fields.limit(5).getInfo()
for feature in features['features']:
  print(feature['properties'])

{'GNDVI': 0.084800905, 'NDVI': 0.060189961, 'NDWI': -0.084800905, 'SAVI': 0.09027955681085587, 'crop_type': 'Rice', 'date_of_image': '01-01-2023', 'field_id': 'Field_1', 'gee_gndvi': 0.2952136856522589, 'gee_ndvi': 0.33055075977730375, 'gee_ndwi': -0.2952136856522589, 'gee_rainfall': 1.2611951306462288, 'gee_savi': 0.1761346600034879, 'gee_temp': 19.472947420392757, 'rainfall': 1.662354196, 'soil_moisture': 46.11935326, 'temperature': 9.884228735, 'yield': 40.21803063}
{'GNDVI': 0.222008568, 'NDVI': 0.21395714, 'NDWI': -0.222008568, 'SAVI': 0.320895526, 'crop_type': 'Rice', 'date_of_image': '16-01-2023', 'field_id': 'Field_1', 'gee_gndvi': 0.4261335943574686, 'gee_ndvi': 0.41055559901458993, 'gee_ndwi': -0.4261335943574686, 'gee_rainfall': 2.446377247571945, 'gee_savi': 0.1975917295945572, 'gee_temp': 19.4976366860526, 'rainfall': 8.446302193, 'soil_moisture': 37.54252546, 'temperature': 13.96707263, 'yield': 30.8703379}
{'GNDVI': 0.431204267, 'NDVI': 0.403306358, 'NDWI': -0.431204267,

In [13]:
sample = ndvi_fields.first()
print(sample.getInfo())
m.add_layer(sample,{'color': 'red'},'Field')
m

{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[88.49792362170211, 22.629728468501135], [88.496562044677, 22.62954938532371], [88.49530885913181, 22.62902639214809], [88.49426382547934, 22.628201122821295], [88.49351013000056, 22.627139273505826], [88.49310776271709, 22.62592537179058], [88.49308874347773, 22.62465604692688], [88.49345457585868, 22.623432337305548], [88.49417613071053, 22.622351647760198], [88.49519596761273, 22.62149999690801], [88.49643290858283, 22.62094517140043], [88.49778849952038, 22.620731331589784], [88.49915484515611, 22.620875497537693], [88.50042319446733, 22.62136619471331], [88.50149259411134, 22.622164367029537], [88.50227792204552, 22.62320648464429], [88.50271666256462, 22.624409599477527], [88.50277388367414, 22.62567794647984], [88.5024450202507, 22.626910565627874], [88.50175624064731, 22.62800933821772], [88.50076236655617, 22.62888679780616], [88.49954251081445, 22.62947309383309], [88.49819377987947, 22.629721553217045], [88

Map(bottom=3653137.0, center=[22.62523109, 88.49792521], controls=(WidgetControl(options=['position', 'transpa…

In [14]:
ndvi_fields.aggregate_array(
    'gee_ndvi'
).getInfo()[:10]

[0.33055075977730375,
 0.41055559901458993,
 0.3998118052458195,
 0.41055559901458993,
 0.41055559901458993,
 0.2139642364229526,
 0.41055559901458993,
 0.41055559901458993,
 0.41055559901458993,
 0.3998118052458195]

In [15]:
ndvi_fields.first().getInfo()

{'type': 'Feature',
 'geometry': {'type': 'Polygon',
  'coordinates': [[[88.49792362170211, 22.629728468501135],
    [88.496562044677, 22.62954938532371],
    [88.49530885913181, 22.62902639214809],
    [88.49426382547934, 22.628201122821295],
    [88.49351013000056, 22.627139273505826],
    [88.49310776271709, 22.62592537179058],
    [88.49308874347773, 22.62465604692688],
    [88.49345457585868, 22.623432337305548],
    [88.49417613071053, 22.622351647760198],
    [88.49519596761273, 22.62149999690801],
    [88.49643290858283, 22.62094517140043],
    [88.49778849952038, 22.620731331589784],
    [88.49915484515611, 22.620875497537693],
    [88.50042319446733, 22.62136619471331],
    [88.50149259411134, 22.622164367029537],
    [88.50227792204552, 22.62320648464429],
    [88.50271666256462, 22.624409599477527],
    [88.50277388367414, 22.62567794647984],
    [88.5024450202507, 22.626910565627874],
    [88.50175624064731, 22.62800933821772],
    [88.50076236655617, 22.62888679780616],
 

In [16]:
import pandas as pd

# Get all features from the ndvi_fields FeatureCollection
all_features = ndvi_fields.getInfo()['features']

# Extract properties from each feature
properties_list = [feature['properties'] for feature in all_features]

# Convert the list of properties to a Pandas DataFrame
ndvi_df = pd.DataFrame(properties_list)

print(ndvi_df.head())

      GNDVI      NDVI      NDWI      SAVI crop_type date_of_image field_id  \
0  0.084801  0.060190 -0.084801  0.090280      Rice    01-01-2023  Field_1   
1  0.222009  0.213957 -0.222009  0.320896      Rice    16-01-2023  Field_1   
2  0.431204  0.403306 -0.431204  0.604837      Rice    31-01-2023  Field_1   
3  0.444132  0.418187 -0.444132  0.627144      Rice    15-02-2023  Field_1   
4  0.387985  0.375138 -0.387985  0.562591      Rice    03-02-2023  Field_1   

   gee_gndvi  gee_ndvi  gee_ndwi  gee_rainfall  gee_savi   gee_temp  \
0   0.295214  0.330551 -0.295214      1.261195  0.176135  19.472947   
1   0.426134  0.410556 -0.426134      2.446377  0.197592  19.497637   
2   0.406732  0.399812 -0.406732      3.722453  0.201092  21.559492   
3   0.426134  0.410556 -0.426134      2.314044  0.197592  19.230342   
4   0.426134  0.410556 -0.426134      1.325544  0.197592  18.863504   

    rainfall  soil_moisture  temperature      yield  
0   1.662354      46.119353     9.884229  40.21803

In [17]:
ndvi_df.to_csv("crop_prediction_gee_ndvi.csv", index = False)